# 02 · Movement labelling

| | |
|---|---|
| **input**  | `data/csv/<subject>/<session>.csv` |
| **output** | `data/formatted/<subject>/<session>.csv` (adds a `label` column) |

The protocol cues one movement per session, but the **onset / offset** of each phase has to be recovered from the signals. Two cues are combined:

* **posture** — hip-marker height thresholds standing vs. sitting;
* **movement onset** — the EMG activation envelope (band-pass → Hilbert →   low-pass) crossing a rest-baseline threshold.

Raw labels are then cleaned: short gaps filled, single-sample flicker removed by a majority filter.

The 7 classes: `stand`, `sit`, `descending`, `ascending`, `walk`, `stand_up`, `sit_down`.

In [ ]:
import sys
from pathlib import Path

# make the `motion_intent` package importable when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from motion_intent import config

In [ ]:
from motion_intent.labeling import (
    emg_envelope, onset_from_envelope, threshold_state,
    fill_short_gaps, majority_smooth, label_transitions,
)

## Load a session

In [ ]:
csv_files = sorted(config.CSV_DIR.rglob('*.csv'))
print(f'{len(csv_files)} session csv files')

df = pd.read_csv(csv_files[0]) if csv_files else None

## Posture from hip height

`Markers_*FTC_*` track the greater trochanter (hip). A per-session height threshold separates the standing and sitting postures.

In [ ]:
HIP_COL = 'Markers_RFTC_Y'      # vertical axis
SIT_STAND_THRESHOLD = 0.75      # metres; tuned per capture volume

if df is not None and HIP_COL in df:
    hip = df[HIP_COL].interpolate()
    posture = threshold_state(hip, SIT_STAND_THRESHOLD, 'stand', 'sit')
    df['posture'] = posture

## Movement onset from EMG

The quadriceps (`emg_RF_*`) burst marks the start of a sit-to-stand or a step. Onsets are matched against posture changes to assign the transition classes (`stand_up`, `sit_down`) and the locomotion classes.

In [ ]:
fs = config.FS
EMG_ONSET_CH = 'EMG_EMG0'       # rename happens in stage 03; raw name here

if df is not None and EMG_ONSET_CH in df:
    env = emg_envelope(df[EMG_ONSET_CH].fillna(0).to_numpy(), fs=fs,
                       band=config.EMG_BAND)
    onsets = onset_from_envelope(env, fs=fs, k=3.0)
    print(f'{len(onsets)} EMG onsets')

## Assemble and clean the label track

Replace the placeholder below with the session-specific rule set (posture segments split by EMG onsets, stair contact from heel markers, etc.). The clean-up steps are generic.

In [ ]:
if df is not None:
    raw_labels = pd.Series(df.get('posture'), index=df.index)   # placeholder

    labels = fill_short_gaps(raw_labels, max_gap=int(0.3 * fs))
    labels = pd.Series(majority_smooth(labels.to_numpy(), win=int(0.2 * fs)),
                       index=df.index)
    df['label'] = labels

    print(df['label'].value_counts(dropna=False))
    print(f"{len(label_transitions(df['label'].to_numpy()))} transitions")

## Save

In [ ]:
for csv_path in csv_files:
    session = pd.read_csv(csv_path)
    # ... apply the labelling rules above to `session` ...
    out_path = config.FORMATTED_DIR / csv_path.parent.name / csv_path.name
    out_path.parent.mkdir(parents=True, exist_ok=True)
    session.to_csv(out_path, index=False)
    print(out_path.relative_to(config.DATA_DIR))